<a href="https://colab.research.google.com/github/SolKacil/matematicas-para-ia/blob/main/python/01-algebra-lineal/02_sistemas_lineales_y_espacio_nulo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 02 &middot; Sistemas lineales: solución general, espacio nulo y rango

**Módulo 1 — Álgebra lineal y geometría diferencial**

Este notebook desarrolla computacionalmente el material de las *Notas de estudio 02*, que siguen a
Deisenroth, Faisal y Ong (2020), *Mathematics for Machine Learning*, capítulo 2, sección 2.3.1.

El objeto de estudio es el sistema lineal $A\mathbf{x} = \mathbf{b}$ con $A \in \mathbb{R}^{m\times n}$.
La pregunta relevante no es únicamente *cuál* es una solución, sino **cuántas** hay y **cómo se
describe el conjunto completo**. La respuesta se organiza en torno a tres objetos —la solución
particular $\mathbf{x}_p$, el espacio nulo $N(A)$ y el rango $\operatorname{rk}(A)$— que se obtienen
todos del mismo procedimiento, la eliminación gaussiana, y que admiten una lectura directa en
modelos de aprendizaje automático, donde $A$ es típicamente una matriz de diseño o una matriz de
pesos.

Una nota sobre la implementación. La eliminación se programa aquí sobre **números racionales
exactos** (`fractions.Fraction`), de modo que los resultados coinciden término a término con los de
las notas y no aparecen artefactos del tipo `0.9999999999999998` donde debe haber un $1$. Al cerrar
cada sección el resultado exacto se contrasta contra la rutina equivalente de NumPy o SciPy, que
opera en punto flotante y mediante algoritmos distintos. La discrepancia entre ambos tratamientos
—álgebra exacta frente a aritmética finita— no es un detalle de implementación: es el origen de la
noción de **rango numérico**, que se discute en la sección 8.

## Al terminar será posible

- Escribir $A\mathbf{x}$ como **combinación lineal de las columnas** de $A$ y emplear esa identidad
  para leer soluciones directamente de la matriz.
- Calcular el **espacio nulo** $N(A)$ y justificar por qué describe todas las soluciones de
  $A\mathbf{x} = \mathbf{b}$ a partir de una sola.
- Determinar la **dependencia o independencia lineal** de un conjunto de vectores y relacionarla con
  la condición $N(A) \neq \{\mathbf{0}\}$.
- Ejecutar la **eliminación gaussiana** hasta las formas escalonada (REF) y escalonada reducida
  (RREF), e identificar variables básicas y libres.
- Obtener una base de $N(A)$ mediante el **método del $-1$** y la inversa de una matriz por
  **Gauss-Jordan**.
- Utilizar el **rango** para decidir si un sistema es consistente, si una matriz es invertible y
  cuántas variables libres tiene la solución general.

## Qué se da por sabido

El notebook [01 · Operaciones básicas](01_operaciones_vectores.ipynb) —producto matriz-vector,
producto de matrices y transpuesta— y lectura de código de Python. No se requiere conocimiento
previo de `sympy` ni de `scipy`.

## Cómo usar este notebook

1. Con el botón **Open in Colab** no se requiere instalación alguna.
2. Las celdas se ejecutan en orden con `Shift + Enter`.
3. Conviene **modificar las matrices de entrada y volver a ejecutar**: salvo indicación contraria,
   las funciones están escritas para una matriz arbitraria y no para el ejemplo concreto.
4. La sección final, **Tu turno**, recoge los ejercicios propuestos de las notas. Sus respuestas
   están en el PDF; el objetivo aquí es reproducirlas y verificarlas con código.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (5.5, 5.5)

print("numpy:", np.__version__)


# --- Utilidades de impresion exacta -------------------------------------------------
# La eliminacion de este notebook opera sobre Fraction, no sobre float. Estas dos
# funciones solo sirven para leer los resultados con comodidad.

def fmt(x):
    """Formatea un racional: entero si el denominador es 1, con barra en caso contrario."""
    x = Fraction(x)
    return str(x.numerator) if x.denominator == 1 else f"{x.numerator}/{x.denominator}"


def mostrar(M, titulo="", corte=None):
    """Imprime una matriz de racionales alineada por columnas.

    `corte` inserta una barra vertical antes de esa columna, para las matrices
    aumentadas [A | b] y [A | I].
    """
    if titulo:
        print(titulo)
    anchos = [max(len(fmt(fila[j])) for fila in M) for j in range(len(M[0]))]
    for fila in M:
        celdas = [fmt(v).rjust(anchos[j]) for j, v in enumerate(fila)]
        if corte is not None:
            celdas.insert(corte, "|")
        print("  [ " + "  ".join(celdas) + " ]")


def a_numpy(M):
    """Convierte una matriz de Fraction en un arreglo de float, para comparar con NumPy."""
    return np.array([[float(x) for x in fila] for fila in M])

---

## 1. Combinación lineal y el producto matriz-vector

Dados $\mathbf{v}_1, \dots, \mathbf{v}_n \in \mathbb{R}^m$ y escalares $c_1, \dots, c_n \in \mathbb{R}$,
se denomina **combinación lineal** de esos vectores a la expresión

$$c_1\mathbf{v}_1 + c_2\mathbf{v}_2 + \dots + c_n\mathbf{v}_n .$$

La observación que organiza todo el tema es que el producto matriz-vector **es** una combinación
lineal de las columnas de la matriz. Si $A$ tiene columnas $\mathbf{c}_1, \dots, \mathbf{c}_n$ y
$\mathbf{x} = [x_1, \dots, x_n]^\top$, entonces

$$A\mathbf{x} = x_1\mathbf{c}_1 + x_2\mathbf{c}_2 + \dots + x_n\mathbf{c}_n .$$

De aquí se sigue la lectura del sistema $A\mathbf{x} = \mathbf{b}$ que se utilizará en todo el
notebook: **resolverlo consiste en determinar con qué coeficientes deben combinarse las columnas de
$A$ para obtener $\mathbf{b}$**. En consecuencia, el sistema admite solución si y sólo si
$\mathbf{b}$ pertenece al subespacio generado por las columnas de $A$.

In [ ]:
v1 = np.array([2.0, 1.0])
v2 = np.array([-1.0, 3.0])

combinacion = 3 * v1 - 2 * v2
print("v1 =", v1, "   v2 =", v2)
print("3*v1 - 2*v2 =", combinacion)

# La misma operacion como producto matriz-vector: v1 y v2 son las columnas de A
# y los coeficientes 3 y -2 son las componentes de x.
A = np.column_stack([v1, v2])
x = np.array([3.0, -2.0])

print("\nA =\n", A)
print("x =", x)
print("A @ x =", A @ x)
print("coinciden:", np.allclose(A @ x, combinacion))

In [ ]:
def combinacion_lineal(columnas, coeficientes):
    """Calcula c_1*v_1 + ... + c_n*v_n sumando termino a termino, sin producto matricial."""
    total = np.zeros_like(np.asarray(columnas[0], dtype=float))
    for v, c in zip(columnas, coeficientes):
        total = total + c * np.asarray(v, dtype=float)
    return total


# La identidad A x = sum_j x_j c_j, verificada sobre una matriz arbitraria.
rng = np.random.default_rng(7)
M = rng.integers(-4, 5, size=(4, 3)).astype(float)
x = np.array([2.0, -1.0, 0.5])
columnas = [M[:, j] for j in range(M.shape[1])]

print("M =\n", M)
print("\nM @ x                        =", M @ x)
print("combinacion de sus columnas  =", combinacion_lineal(columnas, x))
print("iguales:", np.allclose(M @ x, combinacion_lineal(columnas, x)))

### Lectura en aprendizaje automático

En un modelo lineal $\hat{\mathbf{y}} = X\mathbf{w}$, con $X \in \mathbb{R}^{N \times p}$ la matriz
de diseño ($N$ observaciones, $p$ predictores) y $\mathbf{w}$ el vector de coeficientes, el vector
de predicciones es una combinación lineal de las **columnas** de $X$, esto es, de los predictores.
Dos consecuencias inmediatas:

1. El conjunto de vectores de predicción alcanzables por el modelo es exactamente el **espacio
   columna** de $X$. Ajustar el modelo por mínimos cuadrados equivale a proyectar $\mathbf{y}$ sobre
   ese subespacio; el residual es la componente ortogonal a él.
2. Si una columna de $X$ es combinación lineal de las restantes, no amplía el espacio columna: no
   aporta capacidad predictiva, pero sí vuelve no única la solución. Esa situación es la
   **multicolinealidad exacta**, formalizada en la sección 3.

La misma lectura se aplica a una capa densa: en $\mathbf{z} = W\mathbf{x}$ la preactivación es una
combinación lineal de las columnas de $W$, con coeficientes dados por las componentes de la entrada.

In [ ]:
# Matriz de diseno: 6 observaciones y 3 predictores (la primera columna es el intercepto).
X = np.array([[1.0, 2.0, 0.5],
              [1.0, 3.0, 1.5],
              [1.0, 1.0, -0.5],
              [1.0, 4.0, 2.0],
              [1.0, 0.0, 1.0],
              [1.0, 2.5, -1.0]])
w = np.array([0.5, 2.0, -1.0])        # intercepto y dos pendientes

pred = X @ w
print("predicciones =", pred)

# El mismo vector, descompuesto en el aporte de cada predictor:
aportes = [w[j] * X[:, j] for j in range(X.shape[1])]
for j, a in enumerate(aportes):
    print(f"  w[{j}] * columna {j} =", a)
print("suma de los aportes  =", sum(aportes))
print("iguales:", np.allclose(pred, sum(aportes)))

---

## 2. Espacio nulo (kernel)

El **espacio nulo** o **kernel** de $A \in \mathbb{R}^{m \times n}$ es el conjunto de vectores que la
matriz envía al vector cero:

$$N(A) = \ker(A) = \{\, \mathbf{x} \in \mathbb{R}^n : A\mathbf{x} = \mathbf{0} \,\}.$$

Dos propiedades, ambas inmediatas a partir de la definición:

- $\mathbf{0} \in N(A)$ siempre, puesto que $A\mathbf{0} = \mathbf{0}$.
- $N(A)$ es un **subespacio vectorial** de $\mathbb{R}^n$: si $\mathbf{x}_h, \mathbf{x}_h' \in N(A)$,
  entonces $\lambda_1\mathbf{x}_h + \lambda_2\mathbf{x}_h' \in N(A)$ para cualesquiera
  $\lambda_1, \lambda_2 \in \mathbb{R}$.

La relevancia del espacio nulo para el problema $A\mathbf{x} = \mathbf{b}$ es la siguiente. Si
$\mathbf{x}_p$ satisface $A\mathbf{x}_p = \mathbf{b}$ y $\mathbf{x}_h \in N(A)$, entonces

$$A(\mathbf{x}_p + \mathbf{x}_h) = A\mathbf{x}_p + A\mathbf{x}_h = \mathbf{b} + \mathbf{0} = \mathbf{b},$$

de modo que $\mathbf{x}_p + \mathbf{x}_h$ es también solución. Esta identidad es la que permitirá,
en la sección 4, describir el conjunto solución completo a partir de una única solución.

In [ ]:
B = np.array([[1.0, 2.0],
              [2.0, 4.0]])

# Bx = 0 se reduce a x1 + 2*x2 = 0 (la segunda ecuacion es la primera multiplicada por 2),
# de donde x1 = -2*x2 y toda solucion es multiplo de (-2, 1).
generador = np.array([-2.0, 1.0])

print("B =\n", B)
print("\nB @ (-2, 1) =", B @ generador)

# Cerradura del subespacio: cualquier combinacion de elementos de N(B) sigue en N(B).
for t in [-3.0, 0.0, 0.5, 7.0]:
    v = t * generador
    print(f"t = {t:>5} -> v = {v}   B @ v = {B @ v}")

In [ ]:
# La condicion A(xp + xh) = b, verificada numericamente.
b = np.array([3.0, 6.0])          # b = 3*(1,2) esta en el espacio columna de B
xp = np.array([3.0, 0.0])         # solucion particular: 3*c1 + 0*c2

print("B @ xp =", B @ xp, " == b:", np.allclose(B @ xp, b))
print("\nxp + t*(-2, 1) para varios t:")
for t in [-2.0, 0.0, 1.0, 4.5]:
    x = xp + t * generador
    print(f"  t = {t:>5}  x = {x}   B @ x = {B @ x}")

### Lectura en aprendizaje automático

Sea $X$ la matriz de diseño de un modelo lineal. Si $\mathbf{x}_h \in N(X)$ con
$\mathbf{x}_h \neq \mathbf{0}$, entonces para cualquier vector de coeficientes $\mathbf{w}$ se
cumple $X(\mathbf{w} + \mathbf{x}_h) = X\mathbf{w}$: dos vectores de parámetros **distintos**
producen predicciones **idénticas** sobre todo el conjunto de entrenamiento. El modelo no es
identificable, y ningún criterio basado exclusivamente en el error de entrenamiento puede
distinguir entre ambos.

El espacio nulo mide, por tanto, la **redundancia paramétrica** del modelo: sus elementos son
direcciones del espacio de parámetros a lo largo de las cuales la función de pérdida sobre los datos
observados permanece constante. Resolver esa indeterminación exige información adicional —un término
de regularización, una restricción o una parametrización distinta—, tema que se retoma en la
sección 4.

In [ ]:
# Matriz de diseno con un predictor redundante: la tercera columna es la suma
# de las dos primeras (por ejemplo, un total que ya esta desglosado).
Xr = np.array([[1.0, 2.0, 3.0],
               [2.0, 1.0, 3.0],
               [0.0, 4.0, 4.0],
               [3.0, 3.0, 6.0],
               [1.0, 0.0, 1.0]])

# c1 + c2 - c3 = 0, de modo que (1, 1, -1) pertenece a N(Xr).
xh = np.array([1.0, 1.0, -1.0])
print("Xr @ xh =", Xr @ xh)

wa = np.array([0.5, -1.0, 2.0])
wb = wa + 3.0 * xh                 # otro vector de parametros, claramente distinto

print("\nw_a =", wa)
print("w_b =", wb)
print("predicciones con w_a :", Xr @ wa)
print("predicciones con w_b :", Xr @ wb)
print("identicas:", np.allclose(Xr @ wa, Xr @ wb), " <- el modelo no es identificable")

---

## 3. Dependencia e independencia lineal

Un conjunto $\{\mathbf{v}_1, \dots, \mathbf{v}_n\}$ es **linealmente independiente** cuando la única
combinación lineal de sus elementos que produce el vector cero es la trivial:

$$\lambda_1\mathbf{v}_1 + \dots + \lambda_n\mathbf{v}_n = \mathbf{0}
\;\Longrightarrow\; \lambda_1 = \dots = \lambda_n = 0 .$$

Si existe alguna combinación no trivial (con al menos un $\lambda_i \neq 0$) que produce el vector
cero, el conjunto es **linealmente dependiente**.

**Argumento de dimensión.** En $\mathbb{R}^m$ no puede haber más de $m$ vectores linealmente
independientes. Todo conjunto de más de $m$ vectores en $\mathbb{R}^m$ es, por tanto,
necesariamente dependiente.

**Conexión con el espacio nulo.** Para $A$ con columnas $\mathbf{c}_1, \dots, \mathbf{c}_n$,

$$N(A) \neq \{\mathbf{0}\} \iff \{\mathbf{c}_1, \dots, \mathbf{c}_n\} \text{ es linealmente dependiente},$$

puesto que cualquier $\mathbf{x}_h \neq \mathbf{0}$ con $A\mathbf{x}_h = \mathbf{0}$ es, por la
identidad de la sección 1, una combinación no trivial de las columnas que da cero: un testigo
explícito de la dependencia. Cuanto mayor es la dimensión del espacio nulo, más columnas redundantes
posee $A$.

In [ ]:
def son_independientes(vectores):
    """Decide independencia lineal comparando el rango con el numero de vectores."""
    V = np.column_stack([np.asarray(v, dtype=float) for v in vectores])
    return np.linalg.matrix_rank(V) == V.shape[1]


e1, e2 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
print("{e1, e2} independientes:", son_independientes([e1, e2]))

# Las columnas de B (seccion 2) son dependientes: -2*b1 + b2 = 0, y los
# coeficientes (-2, 1) son precisamente el generador de N(B).
b1, b2 = np.array([1.0, 2.0]), np.array([2.0, 4.0])
print("{b1, b2} independientes:", son_independientes([b1, b2]))
print("  -2*b1 + b2 =", -2 * b1 + b2, " <- combinacion no trivial que da cero")

In [ ]:
# Tres vectores en R^2: por el argumento de dimension deben ser dependientes.
w1 = np.array([1.0, 1.0])
w2 = np.array([2.0, 0.0])
w3 = np.array([0.0, 2.0])

print("independientes:", son_independientes([w1, w2, w3]), " (3 vectores en R^2)")

# La combinacion no trivial se obtiene resolviendo el sistema homogeneo. Aqui se
# verifica la que dan las notas; el procedimiento sistematico es el de la seccion 6.
lambdas = np.array([-2.0, 1.0, 1.0])
print("-2*w1 + w2 + w3 =", combinacion_lineal([w1, w2, w3], lambdas))

W = np.column_stack([w1, w2, w3])
print("\nW =\n", W)
print("rango:", np.linalg.matrix_rank(W), " numero de columnas:", W.shape[1])
print("W @ lambdas =", W @ lambdas, " <- el vector de coeficientes esta en N(W)")

### Lectura en aprendizaje automático

La dependencia lineal exacta entre columnas de la matriz de diseño aparece de forma sistemática al
codificar variables categóricas. Si una variable con $k$ categorías se representa con $k$ columnas
indicadoras y el modelo incluye además un término de intercepto, las $k$ indicadoras suman
exactamente la columna de unos: las columnas son dependientes, $N(X) \neq \{\mathbf{0}\}$ y los
coeficientes no quedan determinados de manera única. La práctica habitual —omitir una de las
indicadoras, o prescindir del intercepto— no es una convención arbitraria: es la corrección mínima
que restituye la independencia lineal.

En datos reales lo frecuente no es la dependencia exacta sino la **casi dependencia**: columnas muy
correlacionadas que hacen que $X$ esté mal condicionada. El sistema entonces tiene solución única en
sentido algebraico, pero esa solución es extremadamente sensible a perturbaciones pequeñas de los
datos. La sección 8 retoma el punto bajo la noción de rango numérico.

In [ ]:
# Codificacion indicadora completa de una variable con 3 categorias, mas intercepto.
categorias = np.array([0, 1, 2, 1, 0, 2])
indicadoras = np.eye(3)[categorias]                       # 6 x 3
intercepto = np.ones((len(categorias), 1))
Xcat = np.hstack([intercepto, indicadoras])               # 6 x 4

print("X (intercepto + 3 indicadoras) =\n", Xcat)
print("\ncolumnas:", Xcat.shape[1], " rango:", np.linalg.matrix_rank(Xcat))
print("dependientes:", not son_independientes([Xcat[:, j] for j in range(Xcat.shape[1])]))

# El testigo de la dependencia: las tres indicadoras suman la columna de unos.
testigo = np.array([1.0, -1.0, -1.0, -1.0])
print("Xcat @ (1, -1, -1, -1) =", Xcat @ testigo)

# Correccion habitual: se omite una indicadora (categoria de referencia).
Xref = np.hstack([intercepto, indicadoras[:, 1:]])
print("\nX sin la primera indicadora -> columnas:", Xref.shape[1],
      " rango:", np.linalg.matrix_rank(Xref), " -> independientes")

---

## 4. Solución particular y solución general de $A\mathbf{x} = \mathbf{b}$

Cuando el sistema tiene más incógnitas que ecuaciones ($n > m$) suele admitir infinitas soluciones.
En lugar de enumerarlas, se describen todas mediante

$$\mathbf{x} = \mathbf{x}_p + \mathbf{x}_h, \qquad
A\mathbf{x}_p = \mathbf{b}, \qquad \mathbf{x}_h \in N(A),$$

donde $\mathbf{x}_p$ es **una** solución cualquiera (la solución particular) y $\mathbf{x}_h$ recorre
todo el espacio nulo. El procedimiento consta, por tanto, de tres pasos: obtener una solución
particular, obtener el espacio nulo, y sumarlos.

### Ejemplo

$$\begin{bmatrix} 1 & 0 & 8 & -4 \\ 0 & 1 & 2 & 12 \end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \\ x_4 \end{bmatrix} = \begin{bmatrix} 42 \\ 8 \end{bmatrix}$$

Dos ecuaciones y cuatro incógnitas. Las dos primeras columnas son los vectores canónicos de
$\mathbb{R}^2$, lo que permite leer una solución particular de inmediato:
$\mathbf{b} = 42\,\mathbf{c}_1 + 8\,\mathbf{c}_2$, es decir
$\mathbf{x}_p = (42, 8, 0, 0)^\top$.

Para el espacio nulo se expresan las columnas restantes como combinación de $\mathbf{c}_1$ y
$\mathbf{c}_2$:

$$\mathbf{c}_3 = 8\mathbf{c}_1 + 2\mathbf{c}_2 \;\Longrightarrow\; 8\mathbf{c}_1 + 2\mathbf{c}_2 - \mathbf{c}_3 = \mathbf{0},$$
$$\mathbf{c}_4 = -4\mathbf{c}_1 + 12\mathbf{c}_2 \;\Longrightarrow\; -4\mathbf{c}_1 + 12\mathbf{c}_2 - \mathbf{c}_4 = \mathbf{0},$$

y cada identidad proporciona directamente un elemento de $N(A)$:
$\mathbf{v}_1 = (8, 2, -1, 0)^\top$ y $\mathbf{v}_2 = (-4, 12, 0, -1)^\top$. El signo $-1$ en la
posición de la columna despejada no es casual; es el germen del método general de la sección 6.

In [ ]:
A = np.array([[1.0, 0.0, 8.0, -4.0],
              [0.0, 1.0, 2.0, 12.0]])
b = np.array([42.0, 8.0])

xp = np.array([42.0, 8.0, 0.0, 0.0])
v1 = np.array([8.0, 2.0, -1.0, 0.0])
v2 = np.array([-4.0, 12.0, 0.0, -1.0])

print("A @ xp =", A @ xp, " == b:", np.allclose(A @ xp, b))
print("A @ v1 =", A @ v1)
print("A @ v2 =", A @ v2)

In [ ]:
def solucion_general(xp, base_nucleo, lambdas):
    """Evalua x = xp + sum_i lambda_i v_i para un vector concreto de multiplicadores."""
    x = np.asarray(xp, dtype=float).copy()
    for v, lam in zip(base_nucleo, lambdas):
        x = x + lam * np.asarray(v, dtype=float)
    return x


# Cualquier eleccion de (lambda_1, lambda_2) produce una solucion del sistema.
rng = np.random.default_rng(1)
print("A x = b para 5 elecciones al azar de los multiplicadores:")
for _ in range(5):
    lam = rng.normal(size=2)
    x = solucion_general(xp, [v1, v2], lam)
    print(f"  lambdas = {np.round(lam, 3)}   A @ x = {A @ x}")

### Interpretación geométrica

El conjunto solución de $A\mathbf{x} = \mathbf{b}$ es una **traslación del espacio nulo** por
cualquier solución particular. No contiene al origen —salvo en el caso homogéneo $\mathbf{b} =
\mathbf{0}$— pero es paralelo a $N(A)$ y tiene su misma dimensión. La figura siguiente lo ilustra en
$\mathbb{R}^2$ con un sistema de una sola ecuación, $x_1 + 2x_2 = 4$, cuyo conjunto solución es una
recta y cuyo espacio nulo es la recta paralela que pasa por el origen.

In [ ]:
A2 = np.array([[1.0, 2.0]])
b2 = np.array([4.0])

n2 = np.array([-2.0, 1.0])            # genera N(A2):  1*(-2) + 2*(1) = 0
xp2 = np.array([4.0, 0.0])            # solucion particular con x2 = 0
x_min = np.linalg.lstsq(A2, b2, rcond=None)[0]   # solucion de norma minima

t = np.linspace(-2.2, 2.2, 50)
nulo = np.outer(t, n2)                # recta por el origen
sol = xp2 + np.outer(t, n2)           # recta trasladada

fig, ax = plt.subplots()
ax.plot(nulo[:, 0], nulo[:, 1], "--", color="tab:gray", label=r"$N(A)$:  $Ax = 0$")
ax.plot(sol[:, 0], sol[:, 1], "-", color="tab:blue", label=r"conjunto solucion:  $Ax = b$")
ax.quiver(0, 0, xp2[0], xp2[1], angles="xy", scale_units="xy", scale=1,
          color="tab:red", width=0.011)
ax.scatter(*xp2, color="tab:red", zorder=5)
ax.annotate(r"$x_p = (4, 0)$", xp2, textcoords="offset points", xytext=(6, -14), color="tab:red")
ax.scatter(*x_min, color="tab:green", zorder=5)
ax.annotate("norma minima", x_min, textcoords="offset points", xytext=(8, 6), color="tab:green")
ax.scatter(0, 0, color="black", zorder=5)
ax.annotate("O", (0, 0), textcoords="offset points", xytext=(-14, -4))

ax.set_xlim(-5, 7); ax.set_ylim(-5, 6)
ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
ax.grid(alpha=0.3); ax.set_aspect("equal")
ax.set_title("El conjunto solucion es una traslacion de $N(A)$")
ax.legend(loc="upper right", fontsize=9)
plt.show()

### Lectura en aprendizaje automático

Un sistema con más incógnitas que ecuaciones es la situación característica del **régimen
sobreparametrizado**: más parámetros que observaciones. La descomposición
$\mathbf{x} = \mathbf{x}_p + \mathbf{x}_h$ afirma que existe un subespacio afín completo de
parámetros que reproduce los datos de entrenamiento con error nulo, y que todos sus puntos son
indistinguibles desde el punto de vista del ajuste.

La elección de un punto concreto dentro de ese conjunto es, entonces, una decisión externa al
criterio de ajuste: constituye el **sesgo inductivo** del procedimiento. La elección estándar es la
solución de **norma euclídea mínima**, que es la que devuelven `np.linalg.lstsq` y la pseudoinversa
`np.linalg.pinv`, y que coincide con el límite de la regresión ridge cuando el parámetro de
regularización tiende a cero. Geométricamente es el punto del conjunto solución más próximo al
origen, es decir, el único ortogonal a $N(A)$; en la figura anterior, el punto verde.

In [ ]:
x_min = np.linalg.lstsq(A, b, rcond=None)[0]

print("solucion particular  xp      =", xp, "  norma:", np.linalg.norm(xp))
print("solucion de norma minima     =", x_min, "  norma:", np.linalg.norm(x_min))
print("\nambas satisfacen el sistema:", np.allclose(A @ xp, b), np.allclose(A @ x_min, b))

# La diferencia entre dos soluciones cualesquiera pertenece al espacio nulo.
d = x_min - xp
print("\nx_min - xp =", d)
print("A @ (x_min - xp) =", A @ d, " <- esta en N(A)")

# Y es ortogonal al espacio nulo, que es lo que caracteriza a la de norma minima.
print("\n<x_min, v1> =", float(x_min @ v1), "   <x_min, v2> =", float(x_min @ v2))

---

## 5. Transformaciones elementales y eliminación gaussiana

Los dos ejemplos anteriores se resolvieron por inspección porque la matriz contenía columnas
canónicas. El procedimiento sistemático, válido para cualquier matriz, consiste en llevarla a una
forma más simple mediante **transformaciones elementales de fila**, que no alteran el conjunto
solución:

1. Intercambiar dos filas.
2. Multiplicar una fila por una constante $\lambda \neq 0$.
3. Sumar a una fila un múltiplo de otra.

Las operaciones se aplican sobre la **matriz aumentada** $[A \mid \mathbf{b}]$.

**Forma escalonada (REF).** El *pivote* de una fila es su primer coeficiente no nulo, y debe quedar
estrictamente a la derecha del pivote de la fila anterior; las filas nulas se colocan al final. Ni la
posición de las columnas pivote ni el valor de los pivotes están fijados por la definición: sólo su
orden creciente. **Forma escalonada reducida (RREF).** Además, cada pivote vale $1$ y es la única
entrada no nula de su columna.

Las columnas con pivote determinan las **variables básicas**; las columnas sin pivote, las
**variables libres**. Con $r$ pivotes y $n$ incógnitas hay $n - r$ variables libres, que es
exactamente el número de multiplicadores $\lambda$ en la solución general.

La función siguiente implementa Gauss-Jordan sobre racionales. La elección del pivote prioriza las
entradas $\pm 1$: es una conveniencia para evitar fracciones, no un requisito del método.

In [ ]:
def rref(M, trazar=False):
    """Forma escalonada reducida por Gauss-Jordan, en aritmetica racional exacta.

    Devuelve (R, pivotes), donde `pivotes[i]` es el indice de la columna que
    contiene el pivote de la fila i. Con `trazar=True` imprime cada paso.
    """
    R = [[Fraction(x) for x in fila] for fila in M]
    m, n = len(R), len(R[0])
    pivotes, fila = [], 0

    for col in range(n):
        candidatos = [i for i in range(fila, m) if R[i][col] != 0]
        if not candidatos:                       # columna sin pivote -> variable libre
            continue

        # Transformacion 1: llevar a la posicion del pivote una fila con entrada no nula,
        # prefiriendo +-1 para no introducir fracciones innecesarias.
        i = min(candidatos, key=lambda k: (abs(R[k][col]) != 1, abs(R[k][col])))
        if i != fila:
            R[fila], R[i] = R[i], R[fila]
            if trazar:
                mostrar(R, f"R{fila + 1} <-> R{i + 1}")

        # Transformacion 2: normalizar el pivote a 1.
        p = R[fila][col]
        if p != 1:
            R[fila] = [x / p for x in R[fila]]
            if trazar:
                mostrar(R, f"R{fila + 1} x 1/({fmt(p)})")

        # Transformacion 3: anular el resto de la columna, arriba y abajo.
        for k in range(m):
            if k != fila and R[k][col] != 0:
                factor = R[k][col]
                R[k] = [a - factor * c for a, c in zip(R[k], R[fila])]
        if trazar:
            mostrar(R, f"anular la columna {col + 1} fuera de R{fila + 1}")

        pivotes.append(col)
        fila += 1
        if fila == m:
            break

    return R, pivotes


def rango(M):
    """El rango es el numero de pivotes de la forma escalonada."""
    return len(rref(M)[1])

### Ejemplo con parámetro

Para $a \in \mathbb{R}$ se busca el conjunto solución de

$$\begin{aligned}
-2x_1 + 4x_2 - 2x_3 - x_4 + 4x_5 &= -3 \\
4x_1 - 8x_2 + 3x_3 - 3x_4 + x_5 &= 2 \\
x_1 - 2x_2 + x_3 - x_4 + x_5 &= 0 \\
x_1 - 2x_2 - 3x_4 + 4x_5 &= a
\end{aligned}$$

La eliminación conduce a una última fila de la forma $0 = a + 1$. Se trata de una condición sobre el
parámetro, no sobre las incógnitas: si $a \neq -1$ es una contradicción y el sistema es
**inconsistente**; sólo con $a = -1$ esa fila se reduce a $0 = 0$ y el sistema admite solución.

In [ ]:
def aumentada(A, b):
    return [list(fila) + [bi] for fila, bi in zip(A, b)]


A5 = [[-2, 4, -2, -1, 4],
      [4, -8, 3, -3, 1],
      [1, -2, 1, -1, 1],
      [1, -2, 0, -3, 4]]
b5 = [-3, 2, 0, -1]          # ultimo componente = a, con a = -1

R, piv = rref(aumentada(A5, b5), trazar=True)
mostrar(R, "\nRREF de [A | b] con a = -1:", corte=5)
print("columnas pivote:", [j + 1 for j in piv])

In [ ]:
# Variables basicas y libres se leen de las columnas pivote de A (sin la columna de b).
Ra, piva = rref(A5)
n = len(A5[0])
libres = [j for j in range(n) if j not in piva]

mostrar(Ra, "RREF de A:")
print("\nvariables basicas:", [f"x{j + 1}" for j in piva])
print("variables libres :", [f"x{j + 1}" for j in libres])
print(f"rk(A) = {len(piva)},  incognitas = {n}  ->  {n - len(piva)} variables libres")

In [ ]:
# Consistencia: el sistema tiene solucion si y solo si rk(A) = rk([A | b]).
for a in [-1, 0, 3]:
    b_a = [-3, 2, 0, a]
    r_A = rango(A5)
    r_Ab = rango(aumentada(A5, b_a))
    veredicto = "consistente" if r_A == r_Ab else "sin solucion"
    print(f"a = {a:>3}:  rk(A) = {r_A}   rk([A|b]) = {r_Ab}   -> {veredicto}")

mostrar(rref(aumentada(A5, [-3, 2, 0, 0]))[0],
        "\nRREF de [A | b] con a = 0 (la ultima fila es 0 = 1):", corte=5)

In [ ]:
# Contraste con sympy, que hace exactamente la misma eliminacion exacta.
import sympy as sp

R_sp, piv_sp = sp.Matrix(aumentada(A5, b5)).rref()
print("RREF segun sympy:")
sp.pprint(R_sp)
print("\ncolumnas pivote sympy:", piv_sp, "  propias:", tuple(piv))
print("coinciden:", R_sp.tolist() == [[sp.Rational(x) for x in fila] for fila in R])

### Lectura en aprendizaje automático

La forma escalonada reducida es la herramienta correcta para el análisis **estructural** de un
sistema —consistencia, rango, dimensión del espacio nulo— y por eso se usa aquí con aritmética
exacta. No es, en cambio, el procedimiento con el que se resuelven sistemas en la práctica numérica,
por dos razones.

La primera es de **estabilidad**: la regla de pivoteo empleada arriba minimiza fracciones, pero en
punto flotante la elección del pivote debe hacerse por magnitud máxima (*pivoteo parcial*) para
controlar la amplificación del error de redondeo. La segunda es de **costo**: llevar a RREF una
matriz $n \times n$ requiere $O(n^3)$ operaciones, y ese esfuerzo se desperdicia si se han de
resolver varios sistemas con la misma matriz y distintos lados derechos. Por eso las bibliotecas
numéricas factorizan una sola vez —$LU$ con pivoteo para sistemas cuadrados, $QR$ o SVD para
problemas de mínimos cuadrados— y reutilizan la factorización. En NumPy, `np.linalg.solve` hace
precisamente eso.

In [ ]:
# La solucion exacta por eliminacion coincide con la numerica de np.linalg.solve.
C = [[2, 1, -1],
     [-3, -1, 2],
     [-2, 1, 2]]
d = [8, -11, -3]

Rc, pivc = rref(aumentada(C, d))
mostrar(Rc, "RREF de [C | d]:", corte=3)
x_exacta = [Rc[i][3] for i in range(len(pivc))]
print("solucion exacta :", [fmt(x) for x in x_exacta])
print("np.linalg.solve :", np.linalg.solve(np.array(C, dtype=float), np.array(d, dtype=float)))

---

## 6. El espacio nulo por el método del $-1$

Con la matriz en forma escalonada reducida, el espacio nulo puede obtenerse por sustitución hacia
atrás: se fija una variable libre en $1$, las demás en $0$, y se resuelve el sistema homogéneo. El
procedimiento es correcto pero poco sistemático. El **método del $-1$** lo reemplaza por una
construcción puramente mecánica.

**Método.** Sea $A \in \mathbb{R}^{k \times n}$ en RREF y sin filas nulas, con columnas pivote
$j_1, \dots, j_k$.

1. Se completa $A$ a una matriz cuadrada $\tilde{A} \in \mathbb{R}^{n \times n}$ insertando, por cada
   variable libre $x_j$, una fila $[0 \cdots 0 \; -1 \; 0 \cdots 0]$ con el $-1$ exactamente en la
   columna $j$. La diagonal de $\tilde{A}$ queda entonces con un $1$ en cada columna pivote y un
   $-1$ en cada columna libre.
2. Cada columna de $\tilde{A}$ que tiene el $-1$ en la diagonal es una solución de
   $A\mathbf{x} = \mathbf{0}$, y esas columnas forman una **base** de $N(A)$.

**Justificación.** La fila del pivote de $x_{j_i}$ expresa
$x_{j_i} + \sum_{j \text{ libre}} r_j x_j = 0$, donde $r_j$ es la entrada de la RREF en la columna
libre $j$. Al fijar esa variable libre en $-1$ y las restantes en $0$ se obtiene $x_{j_i} = r_j$,
que es precisamente la entrada ya presente en la matriz. De ahí que baste copiar la columna libre y
colocar un $-1$ en su propia posición, sin sustituir nada: fijar la variable libre en $-1$ en vez de
en $+1$ es lo que permite leer los coeficientes directamente.

In [ ]:
def nucleo_menos_uno(M):
    """Base del espacio nulo de M por el metodo del -1.

    Devuelve (A_tilde, base, pivotes, libres), con `base` como lista de vectores
    de Fraction: las columnas de A_tilde cuyo elemento diagonal es -1.
    """
    R, pivotes = rref(M)
    n = len(M[0])
    libres = [j for j in range(n) if j not in pivotes]

    At = [[Fraction(0)] * n for _ in range(n)]
    for i, j in enumerate(pivotes):      # la fila del pivote j_i se coloca en la fila j_i
        At[j] = R[i][:]
    for j in libres:                     # y cada variable libre aporta su fila con -1
        fila = [Fraction(0)] * n
        fila[j] = Fraction(-1)
        At[j] = fila

    base = [[At[i][j] for i in range(n)] for j in libres]   # columnas libres de A_tilde
    return At, base, pivotes, libres

In [ ]:
# Aplicado al ejemplo de la seccion 5 (la matriz A5, con a = -1).
At, base, piv5, libres5 = nucleo_menos_uno(A5)

mostrar(At, "A tilde (5 x 5):")
print("\ncolumnas libres:", [j + 1 for j in libres5], "-> base del espacio nulo:")
for k, v in enumerate(base, start=1):
    print(f"  v'{k} = ({', '.join(fmt(x) for x in v)})")

An = np.array(A5, dtype=float)
for k, v in enumerate(base, start=1):
    print(f"\nA @ v'{k} =", An @ np.array([float(x) for x in v]))

In [ ]:
# Comparacion con la sustitucion directa: fijar una variable libre en 1 y el resto en 0.
v1_sust = np.array([2.0, 1.0, 0.0, 0.0, 0.0])      # con x2 = 1, x5 = 0
v2_sust = np.array([2.0, 0.0, -1.0, 2.0, 1.0])     # con x2 = 0, x5 = 1

print("A @ v1 =", An @ v1_sust, "   A @ v2 =", An @ v2_sust)

B_menos_uno = np.array([[float(x) for x in v] for v in base]).T
B_sust = np.column_stack([v1_sust, v2_sust])

print("\nbase por el metodo del -1:\n", B_menos_uno)
print("\nbase por sustitucion directa:\n", B_sust)
print("\nv' = -v en ambos casos:", np.allclose(B_menos_uno, -B_sust))

Las dos bases difieren en un factor $-1$. No es un error: el espacio nulo es un subespacio, de modo
que cualquier múltiplo escalar no nulo de un generador sigue siendo un generador válido. Lo que
identifica al espacio nulo no es una base concreta sino el subespacio que ésta genera, y ambas
generan el mismo. La solución general se escribe igual con cualquiera de las dos, cambiando sólo el
signo de los multiplicadores.

La verificación de que dos conjuntos de vectores generan el mismo subespacio se hace comparando
rangos: si $\operatorname{rk}(B_1) = \operatorname{rk}(B_2) = \operatorname{rk}([B_1 \mid B_2])$,
entonces $\operatorname{span}(B_1) = \operatorname{span}(B_2)$. La comparación se realiza abajo
contra la base que devuelve `scipy.linalg.null_space`, obtenida por descomposición en valores
singulares y por tanto **ortonormal**, distinta de la exacta.

In [ ]:
from scipy.linalg import null_space

Ns = null_space(An)          # base ortonormal calculada por SVD

print("base ortonormal de SciPy (columnas):\n", Ns)
print("\nA @ Ns =\n", An @ Ns)

# Si ambas bases generan el mismo subespacio, apilarlas no aumenta el rango.
# Los valores singulares de la matriz conjunta muestran donde esta el corte: dos
# valores de orden 1 y dos del orden del error de redondeo. La tolerancia por
# defecto de matrix_rank cae justo en esa frontera, asi que conviene fijarla.
conjunta = np.hstack([B_menos_uno, Ns])
print("\nvalores singulares de [base propia | base de SciPy]:")
print(" ", np.array2string(np.linalg.svd(conjunta, compute_uv=False),
                           formatter={"float_kind": lambda x: f"{x:.2e}"}))

tol = 1e-10
print(f"\ncon tol = {tol:g}:  rk(propia) = {np.linalg.matrix_rank(B_menos_uno, tol=tol)}"
      f"   rk(scipy) = {np.linalg.matrix_rank(Ns, tol=tol)}"
      f"   rk(juntas) = {np.linalg.matrix_rank(conjunta, tol=tol)}")
print("generan el mismo subespacio:",
      np.linalg.matrix_rank(conjunta, tol=tol) == np.linalg.matrix_rank(B_menos_uno, tol=tol))

### Solución general del ejemplo

Reuniendo la solución particular de la sección 5 y la base del espacio nulo:

$$\left\{ \mathbf{x} \in \mathbb{R}^5 :
\mathbf{x} = \begin{bmatrix} 2 \\ 0 \\ -1 \\ 1 \\ 0 \end{bmatrix}
+ \lambda_1 \begin{bmatrix} 2 \\ 1 \\ 0 \\ 0 \\ 0 \end{bmatrix}
+ \lambda_2 \begin{bmatrix} 2 \\ 0 \\ -1 \\ 2 \\ 1 \end{bmatrix},
\;\; \lambda_1, \lambda_2 \in \mathbb{R} \right\}$$

La solución particular se lee de la RREF de $[A \mid \mathbf{b}]$ asignando $0$ a las variables
libres: las variables básicas toman entonces los valores de la última columna.

In [ ]:
def solucion_particular(A, b):
    """Solucion con todas las variables libres fijadas en 0, leida de la RREF."""
    R, piv_ab = rref(aumentada(A, b))
    n = len(A[0])
    if piv_ab and piv_ab[-1] == n:                     # pivote en la columna de b
        raise ValueError("sistema inconsistente: rk(A) != rk([A|b])")
    x = [Fraction(0)] * n
    for i, j in enumerate(piv_ab):
        x[j] = R[i][n]
    return x


xp5 = solucion_particular(A5, b5)
print("xp =", [fmt(x) for x in xp5])
print("A @ xp =", An @ np.array([float(x) for x in xp5]), "   b =", b5)

# El conjunto solucion completo, muestreado.
xp_np = np.array([float(x) for x in xp5])
print("\nA x = b para varias elecciones de (lambda_1, lambda_2):")
for lam in [(0.0, 0.0), (1.0, 0.0), (0.0, 1.0), (-2.5, 3.0)]:
    x = solucion_general(xp_np, [v1_sust, v2_sust], lam)
    print(f"  lambdas = {lam}   A @ x = {An @ x}")

# Y la verificacion de que el sistema inconsistente es rechazado.
try:
    solucion_particular(A5, [-3, 2, 0, 0])
except ValueError as e:
    print("\ncon a = 0 ->", e)

### Lectura en aprendizaje automático

Conviene distinguir las dos bases del espacio nulo que se han calculado, porque responden a
propósitos distintos. La del método del $-1$ es **exacta y dispersa**: sus vectores tienen ceros en
las posiciones de las demás variables libres, lo que permite interpretar cada uno como una relación
de dependencia concreta entre columnas —qué predictor es redundante y respecto a cuáles—. La de
`scipy.linalg.null_space` es **ortonormal** y se obtiene de la descomposición en valores singulares;
no es interpretable columna a columna, pero es numéricamente estable y, sobre todo, es la única
aplicable cuando las entradas son datos medidos y la dependencia es aproximada en lugar de exacta.

Esa distinción es la que separa el álgebra lineal exacta del cálculo matricial numérico: con datos
reales, $N(X)$ es casi siempre $\{\mathbf{0}\}$ en sentido estricto, y la pregunta pertinente pasa a
ser qué direcciones están *cerca* del espacio nulo. La sección 8 la responde con los valores
singulares.

---

## 7. Cálculo de la inversa por eliminación de Gauss-Jordan

Para $A \in \mathbb{R}^{n \times n}$, encontrar $A^{-1}$ significa hallar $X$ con $AX = I_n$. Puesto
que $X = [\mathbf{x}_1 \mid \cdots \mid \mathbf{x}_n]$, se trata de $n$ sistemas lineales con la
misma matriz de coeficientes y distintos lados derechos: las columnas de $I_n$. Los $n$ sistemas se
resuelven simultáneamente llevando la matriz aumentada $[A \mid I_n]$ a su forma escalonada
reducida, con lo que el bloque derecho se transforma en la inversa:

$$[A \mid I_n] \;\rightsquigarrow\; \cdots \;\rightsquigarrow\; [I_n \mid A^{-1}].$$

Si durante la eliminación aparece una fila nula en el bloque izquierdo, el proceso se detiene: eso
significa $\operatorname{rk}(A) < n$ y la matriz no es invertible.

In [ ]:
def inversa_gauss_jordan(A):
    """Inversa exacta por Gauss-Jordan sobre [A | I]; error si A es singular."""
    n = len(A)
    if any(len(fila) != n for fila in A):
        raise ValueError("la matriz debe ser cuadrada")

    ident = [[Fraction(int(i == j)) for j in range(n)] for i in range(n)]
    aug = [list(fila) + ident[i] for i, fila in enumerate(A)]

    R, pivotes = rref(aug)
    if pivotes[:n] != list(range(n)):
        raise ValueError(f"matriz singular: rk(A) = {len([j for j in pivotes if j < n])} < {n}")
    return [[R[i][n + j] for j in range(n)] for i in range(n)], R

In [ ]:
G = [[1, 0, 2, 0],
     [1, 1, 0, 0],
     [1, 2, 0, 1],
     [1, 1, 1, 1]]

inv, R_aug = inversa_gauss_jordan(G)
mostrar(R_aug, "[A | I] reducida a [I | A^-1]:", corte=4)
mostrar(inv, "\nA^-1 =")

# Verificacion exacta: el producto A A^-1 se calcula tambien con racionales.
producto = [[sum(Fraction(G[i][k]) * inv[k][j] for k in range(4)) for j in range(4)]
            for i in range(4)]
mostrar(producto, "\nA @ A^-1 =")

print("\nnp.linalg.inv:\n", np.linalg.inv(np.array(G, dtype=float)))
print("coincide con la exacta:", np.allclose(np.linalg.inv(np.array(G, dtype=float)), a_numpy(inv)))

In [ ]:
# Una matriz singular detiene el procedimiento: rk(B) = 1 < 2.
try:
    inversa_gauss_jordan([[1, 2], [2, 4]])
except ValueError as e:
    print("[[1,2],[2,4]] ->", e)

mostrar(rref([[1, 2, 1, 0], [2, 4, 0, 1]])[0],
        "\n[B | I] reducida: la fila de ceros a la izquierda es rk(B) < n:", corte=2)

### Lectura en aprendizaje automático

La inversa aparece en la solución de forma cerrada de los mínimos cuadrados ordinarios, dada por las
ecuaciones normales $\mathbf{w} = (X^\top X)^{-1} X^\top \mathbf{y}$. La expresión es correcta como
identidad algebraica, pero **no es la manera de calcularla**. Formar explícitamente la inversa cuesta
más operaciones que resolver el sistema, y sobre todo destruye precisión: el número de condición de
$X^\top X$ es el cuadrado del de $X$, de modo que el planteamiento por ecuaciones normales duplica la
pérdida de dígitos significativos frente a una factorización $QR$ o una SVD aplicadas directamente a
$X$.

La regla práctica es explícita: `np.linalg.solve(A, b)` en lugar de `np.linalg.inv(A) @ b`, y
`np.linalg.lstsq(X, y)` en lugar de las ecuaciones normales. La celda siguiente cuantifica la
diferencia sobre una matriz de Hilbert, mal condicionada por construcción.

In [ ]:
from scipy.linalg import hilbert

n = 10
H = hilbert(n)                      # mal condicionada por construccion
x_real = np.ones(n)
rhs = H @ x_real

x_inv = np.linalg.inv(H) @ rhs      # con inversa explicita
x_solve = np.linalg.solve(H, rhs)   # con factorizacion LU

print(f"numero de condicion de H: {np.linalg.cond(H):.3e}")
print(f"error con inv(H) @ b   : {np.linalg.norm(x_inv - x_real):.3e}")
print(f"error con solve(H, b)  : {np.linalg.norm(x_solve - x_real):.3e}")

# El mismo contraste en la regresion: ecuaciones normales frente a lstsq.
X = hilbert(10)[:, :6]
y = X @ np.ones(6)

w_normales = np.linalg.inv(X.T @ X) @ X.T @ y
w_lstsq = np.linalg.lstsq(X, y, rcond=None)[0]

print(f"\ncond(X)     = {np.linalg.cond(X):.3e}")
print(f"cond(X^T X) = {np.linalg.cond(X.T @ X):.3e}   <- aproximadamente el cuadrado")
print(f"error con ecuaciones normales: {np.linalg.norm(w_normales - 1.0):.3e}")
print(f"error con lstsq              : {np.linalg.norm(w_lstsq - 1.0):.3e}")

---

## 8. Rango de una matriz

**Definición.** El número de columnas linealmente independientes de $A \in \mathbb{R}^{m \times n}$
coincide con el número de filas linealmente independientes, y a ese número se le llama **rango** de
$A$, denotado $\operatorname{rk}(A)$.

La igualdad entre el rango por filas y el rango por columnas no es evidente a priori, y es uno de
los resultados centrales del tema. Operativamente, el rango se obtiene por eliminación gaussiana:
es el número de filas no nulas de la forma escalonada, o equivalentemente el número de pivotes,
que es lo mismo que ya se contaba como número de variables básicas.

El rango responde, por sí solo, a las cuatro preguntas estructurales del tema:

1. **Cuántas variables libres hay.** $\dim N(A) = n - \operatorname{rk}(A)$, con $n$ el número de
   columnas. Esta identidad es el **teorema del rango-nulidad**.
2. **Si una matriz cuadrada es invertible.** $A \in \mathbb{R}^{n\times n}$ es invertible si y sólo
   si $\operatorname{rk}(A) = n$.
3. **Si el sistema es consistente.** $A\mathbf{x} = \mathbf{b}$ tiene solución si y sólo si
   $\operatorname{rk}(A) = \operatorname{rk}([A \mid \mathbf{b}])$.
4. **Si la matriz es de rango completo.** $A$ tiene rango completo cuando
   $\operatorname{rk}(A) = \min(m, n)$, el máximo posible para su tamaño; en caso contrario es
   **deficiente en rango**, término que para matrices cuadradas equivale a *singular*.

In [ ]:
ejemplos = {
    "ya escalonada": [[1, 0, 1], [0, 1, 1], [0, 0, 0]],
    "con eliminacion": [[1, 2, 1], [-2, -3, 1], [3, 5, 0]],
    "singular 2x2": [[1, 2], [2, 4]],
    "A5 (seccion 5)": A5,
}

for nombre, M in ejemplos.items():
    R, piv = rref(M)
    r_np = np.linalg.matrix_rank(np.array(M, dtype=float))
    n_col = len(M[0])
    print(f"{nombre:>18}: rk = {len(piv)}  (numpy: {r_np})   "
          f"dim N = {n_col - len(piv)}   columnas = {n_col}")

mostrar(rref(ejemplos["con eliminacion"])[0], "\nRREF del segundo ejemplo (2 filas no nulas):")

In [ ]:
# Teorema del rango-nulidad, verificado sobre el ejemplo de la seccion 5.
_, base5, piv5, libres5 = nucleo_menos_uno(A5)
n_col = len(A5[0])

print(f"n = {n_col}   rk(A) = {len(piv5)}   dim N(A) = {len(base5)}")
print(f"n - rk(A) = {n_col - len(piv5)} = dim N(A):", n_col - len(piv5) == len(base5))
print("null_space de SciPy devuelve", null_space(An).shape[1], "vectores")

In [ ]:
# Rango e invertibilidad, sobre matrices aleatorias y sobre una construida deficiente.
rng = np.random.default_rng(3)
P = rng.normal(size=(5, 5))
Q = P.copy()
Q[:, 4] = Q[:, 0] + 2 * Q[:, 1]        # una columna dependiente de las otras

for nombre, M in [("P generica", P), ("Q deficiente", Q)]:
    r = np.linalg.matrix_rank(M)
    print(f"{nombre:>14}: rk = {r}   rango completo: {r == 5}   "
          f"|det| = {abs(np.linalg.det(M)):.3e}")

try:
    np.linalg.inv(Q)
except np.linalg.LinAlgError as e:
    print("\ninv(Q) ->", e)
print("dim N(Q) =", null_space(Q).shape[1])

### Rango numérico

Con datos medidos, la dependencia lineal exacta es excepcional: basta una perturbación mínima en una
entrada para que una matriz deficiente pase a tener rango completo, aunque siga comportándose a
todos los efectos como si no lo tuviera. El rango algebraico es, en ese régimen, una cantidad
discontinua e inservible.

La noción operativa es el **rango numérico**, definido a partir de los valores singulares
$\sigma_1 \geq \sigma_2 \geq \dots \geq \sigma_{\min(m,n)} \geq 0$ de $A$ como el número de ellos que
superan una tolerancia $\tau$. Esto es lo que calcula `np.linalg.matrix_rank`, cuya tolerancia por
defecto es $\sigma_1 \cdot \max(m, n) \cdot \varepsilon$, con $\varepsilon$ el épsilon de máquina.
Un valor singular nulo corresponde a una dirección del espacio nulo; un valor singular pequeño, a
una dirección *casi* nula a lo largo de la cual la matriz comprime severamente y, al invertir,
amplifica el error. El cociente $\sigma_1/\sigma_r$ es el número de condición que aparecía en la
sección 7.

In [ ]:
# Matriz de rango 3 por construccion, perturbada con ruido de distintas magnitudes.
rng = np.random.default_rng(11)
M0 = rng.normal(size=(60, 3)) @ rng.normal(size=(3, 8))     # rk exacto = 3

cientifico = {"float_kind": lambda x: f"{x:.2e}"}
PISO = 1e-16          # los valores exactamente nulos se dibujan en el piso de la escala

fig, ax = plt.subplots(figsize=(7.2, 4.4))
print(f"{'ruido':>8} | {'rk (tol por defecto)':>21} | {'rk (tol = 1e-3 * s1)':>21}")
print("-" * 56)

for ruido, color in [(0.0, "tab:blue"), (1e-10, "tab:green"),
                     (1e-6, "tab:orange"), (1e-2, "tab:red")]:
    M = M0 + ruido * rng.normal(size=M0.shape)
    s = np.linalg.svd(M, compute_uv=False)
    rk_def = np.linalg.matrix_rank(M)
    rk_rel = np.linalg.matrix_rank(M, tol=1e-3 * s[0])
    print(f"{ruido:>8g} | {rk_def:>21} | {rk_rel:>21}")
    print(f"           s = {np.array2string(s, formatter=cientifico)}")
    ax.semilogy(range(1, len(s) + 1), np.maximum(s, PISO), "o-", color=color,
                label=f"ruido = {ruido:g}")

ax.set_xlabel("indice del valor singular")
ax.set_ylabel(r"$\sigma_i$  (escala logaritmica)")
ax.set_ylim(PISO / 10, 1e2)
ax.set_title("Los valores singulares revelan el rango efectivo")
ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.show()

La tabla anterior muestra por qué la tolerancia no es un detalle. Con el criterio por defecto basta
un ruido de $10^{-10}$ —irrelevante frente a valores singulares del orden de $10$— para que la
matriz pase a tener rango $8$: formalmente correcto, e inútil como descripción de los datos. Con una
tolerancia **relativa** al mayor valor singular, en cambio, el rango se mantiene en $3$ mientras la
perturbación sea pequeña comparada con la señal, y sólo cambia cuando el ruido alcanza una magnitud
comparable a la de las direcciones que se quiere conservar. La elección de $\tau$ es, en definitiva,
una decisión sobre qué magnitud se considera señal y cuál ruido; no la determina el álgebra.

### Lectura en aprendizaje automático

El rango cuantifica la **dimensión efectiva** de la información contenida en una matriz, y esa
lectura sostiene varias técnicas de uso corriente:

- **Análisis de componentes principales.** Retener las $k$ direcciones de mayor valor singular
  equivale a sustituir la matriz de datos por la mejor aproximación de rango $k$ en norma de
  Frobenius (teorema de Eckart-Young). El decaimiento de los valores singulares indica cuánta
  redundancia admite el conjunto de datos.
- **Adaptación de bajo rango.** Métodos como LoRA ajustan un modelo preentrenado añadiendo a cada
  matriz de pesos una corrección $\Delta W = BA$ con $B \in \mathbb{R}^{d \times r}$,
  $A \in \mathbb{R}^{r \times d}$ y $r \ll d$; el rango $r$ es, literalmente, el presupuesto de
  parámetros del ajuste.
- **Diagnóstico de la matriz de diseño.** Una matriz de diseño deficiente en rango produce
  coeficientes no identificables (sección 2); una casi deficiente produce coeficientes inestables,
  con varianza muy alta, que cambian de signo ante perturbaciones mínimas de los datos. Inspeccionar
  los valores singulares de $X$ antes de ajustar detecta ambas situaciones.

In [ ]:
# Aproximacion de rango k y el error que introduce (Eckart-Young).
rng = np.random.default_rng(5)
D = rng.normal(size=(40, 40)) @ rng.normal(size=(40, 5)) @ rng.normal(size=(5, 40))

U, s, Vt = np.linalg.svd(D, full_matrices=False)
print("rango numerico de D:", np.linalg.matrix_rank(D))

for k in [1, 3, 5, 8]:
    Dk = (U[:, :k] * s[:k]) @ Vt[:k]
    error = np.linalg.norm(D - Dk) / np.linalg.norm(D)
    parametros = k * (D.shape[0] + D.shape[1])
    print(f"  k = {k}:  error relativo = {error:.2e}   "
          f"parametros = {parametros:>5} de {D.size}")

---

## Tu turno

Los ejercicios son los propuestos en las *Notas de estudio 02*. Sus respuestas están en el PDF, de
modo que el objetivo aquí no es adivinarlas sino **reproducirlas con código y verificarlas**: cada
resultado debe comprobarse con `np.allclose`, con `A @ v` o con la función exacta correspondiente de
este notebook. En los ejercicios de espacio nulo se pide emplear el método del $-1$.

**1. Combinación lineal.**
&nbsp;&nbsp;a) Con $\mathbf{v}_1 = (3, -1)^\top$ y $\mathbf{v}_2 = (2, 5)^\top$, calcular
$4\mathbf{v}_1 - 2\mathbf{v}_2$.
&nbsp;&nbsp;b) Con $A = \begin{bmatrix} 1 & 4 \\ 2 & -3 \end{bmatrix}$ y $\mathbf{x} = (3, 1)^\top$,
calcular $A\mathbf{x}$ de las dos maneras —producto matriz-vector y combinación lineal de las
columnas— y verificar que coinciden.

**2. Espacio nulo por el método del $-1$.** Obtener $N(B)$ para

$$B_1 = \begin{bmatrix} 1 & 2 & 0 \\ 0 & 0 & 1 \end{bmatrix}, \qquad
B_2 = \begin{bmatrix} 1 & 0 & 2 & 3 \\ 0 & 1 & -1 & 4 \end{bmatrix}$$

(ambas ya en RREF). Construir $\tilde{B}$ a mano en una celda, comparar con la salida de
`nucleo_menos_uno` y comprobar que $B\mathbf{v} = \mathbf{0}$ para cada vector de la base.

**3. Dependencia lineal.**
&nbsp;&nbsp;a) ¿Son $\mathbf{v}_1 = (2, 1)$ y $\mathbf{v}_2 = (1, 3)$ linealmente independientes?
&nbsp;&nbsp;b) ¿Lo son $\mathbf{u}_1 = (1, -1)$, $\mathbf{u}_2 = (2, 1)$, $\mathbf{u}_3 = (-1, 4)$?
En el caso dependiente, obtener una combinación no trivial que dé el vector cero **calculándola**
con `nucleo_menos_uno` sobre la matriz cuyas columnas son esos vectores, en lugar de proponerla.

**4. Solución general.** Resolver $A\mathbf{x} = \mathbf{b}$ para

$$A_1 = \begin{bmatrix} 1 & 0 & 3 & -2 \\ 0 & 1 & 1 & 5 \end{bmatrix},\;
\mathbf{b}_1 = \begin{bmatrix} 10 \\ 4 \end{bmatrix}; \qquad
A_2 = \begin{bmatrix} 1 & 0 & 2 & -1 & 3 \\ 0 & 1 & 4 & 2 & -5 \end{bmatrix},\;
\mathbf{b}_2 = \begin{bmatrix} 7 \\ -3 \end{bmatrix}$$

Escribir la solución general, muestrear diez valores aleatorios de los multiplicadores y comprobar
que todos satisfacen el sistema. Comparar después $\mathbf{x}_p$ con la solución que devuelve
`np.linalg.lstsq` y verificar que la diferencia pertenece al espacio nulo.

**5. Eliminación gaussiana.** Resolver por eliminación, mostrando la traza con `trazar=True`:
&nbsp;&nbsp;a) $x_1 + x_2 + x_3 = 6$, $\;2x_1 + 3x_2 - x_3 = 5$.
&nbsp;&nbsp;b) $x_1 + x_2 + 2x_3 + x_4 = 5$, $\;2x_1 + x_2 + 3x_3 - x_4 = 4$,
$\;x_1 - x_2 - x_3 + 2x_4 = 3$.
Identificar en cada caso las variables básicas y las libres antes de leer la solución.

**6. Inversa.** Calcular $A^{-1}$ con `inversa_gauss_jordan` y verificar $AA^{-1} = I$ para

$$A_1 = \begin{bmatrix} 2 & 1 \\ 1 & 1 \end{bmatrix}, \qquad
A_2 = \begin{bmatrix} 1 & -2 & 1 \\ 1 & -2 & 0 \\ 0 & -1 & -1 \end{bmatrix}$$

**7. Extensión.** Escribir una función `describir(A, b)` que, dada una matriz y un lado derecho,
imprima el rango de $A$, el rango de $[A \mid \mathbf{b}]$, el número de variables libres y, según
el caso, el mensaje «sin solución», «solución única» o la solución general completa. Probarla con
todos los sistemas de los ejercicios anteriores.

In [ ]:
# Tu codigo aqui.
v1 = np.array([3.0, -1.0])
v2 = np.array([2.0, 5.0])

# 1a.

---

## Resumen

| Concepto | Definición | Cómo se calcula aquí | Equivalente numérico |
|---|---|---|---|
| Combinación lineal | $\sum_i c_i \mathbf{v}_i$ | `combinacion_lineal` | `A @ x` |
| Espacio nulo | $\{\mathbf{x} : A\mathbf{x} = \mathbf{0}\}$ | `nucleo_menos_uno` | `scipy.linalg.null_space` |
| Independencia lineal | sólo la combinación trivial da $\mathbf{0}$ | $N(A) = \{\mathbf{0}\}$ | `matrix_rank == n_columnas` |
| Forma escalonada reducida | pivotes unitarios y aislados | `rref` | `sympy.Matrix(...).rref()` |
| Solución particular | variables libres en $0$ | `solucion_particular` | `np.linalg.lstsq` (norma mínima) |
| Solución general | $\mathbf{x}_p + \mathbf{x}_h$, $\mathbf{x}_h \in N(A)$ | `solucion_general` | — |
| Inversa | $AX = I$ | `inversa_gauss_jordan` | `np.linalg.inv`, `np.linalg.solve` |
| Rango | pivotes de la forma escalonada | `rango` | `np.linalg.matrix_rank` (SVD) |

Cuatro ideas para retener:

1. **Todo el tema es una sola identidad.** $A\mathbf{x}$ es una combinación lineal de las columnas
   de $A$; resolver, decidir dependencia, calcular el núcleo y determinar el rango son lecturas
   distintas de esa misma expresión.
2. **La solución nunca es un punto, es un conjunto.** La forma $\mathbf{x}_p + N(A)$ vale siempre;
   la solución única es el caso particular $N(A) = \{\mathbf{0}\}$.
3. **El rango decide todo lo estructural.** Consistencia, invertibilidad y número de variables
   libres se leen del rango, y éste se obtiene contando pivotes.
4. **La aritmética exacta y la numérica responden preguntas distintas.** La RREF es la herramienta
   del análisis estructural; con datos medidos, la pregunta pertinente es el rango numérico, que se
   lee de los valores singulares.

## Qué sigue

- La versión en script, más corta y sin explicaciones: [`02_sistemas_lineales_y_espacio_nulo.py`](02_sistemas_lineales_y_espacio_nulo.py)
- Las notas de la sesión: [`Notas_de_estudio_02.pdf`](Notas_de_estudio_02.pdf)
- El notebook anterior: [`01 · Operaciones básicas con vectores y matrices`](01_operaciones_vectores.ipynb)
- Los demás módulos, en el [README del repositorio](../../README.md)

**Referencia.** Deisenroth, M. P., Faisal, A. A. y Ong, C. S. (2020). *Mathematics for Machine
Learning*. Cambridge University Press, capítulo 2, sección 2.3.1.

---

*Material abierto bajo licencia MIT. ¿Encontraste un error o quieres aportar un ejercicio?
Lee [CONTRIBUTING.md](../../CONTRIBUTING.md).*